In [2]:
import tensorflow as tf
from tensorflow import keras


In [3]:
import keras_tuner as kt


In [4]:
(img_train, label_train), (img_test, label_test) = keras.datasets.fashion_mnist.load_data()


In [5]:
img_train = img_train.astype('float32') / 255.0
img_test = img_test.astype('float32') / 255.0

In [9]:
def model_builder(hp):
  model = keras.Sequential()
  model.add(keras.layers.Flatten(input_shape=(28, 28)))

  # Tune the number of units in the first Dense layer
  # Choose an optimal value between 32-512
  hp_units = hp.Int('units', min_value=32, max_value=512, step=32)
  model.add(keras.layers.Dense(units=hp_units, activation='relu'))
  model.add(keras.layers.Dense(10))

  # Tune the learning rate for the optimizer
  # Choose an optimal value from 0.01, 0.001, or 0.0001
  hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

  model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy'])

  return model



In [11]:
tuner = kt.Hyperband(model_builder,
                     objective='val_accuracy',
                     max_epochs=10,
                     factor=3,
                     directory='my_dir',
                     project_name='intro_to_kt')


In [15]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience = 5)

In [16]:
tuner.search(img_train, label_train, epochs=50, validation_split=0.2, callbacks=[stop_early])

#Get the optimal hyperparameters 
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The hyperparameter search is complete. The optimal number of units in the first densely-connected
layer is {best_hps.get('units')} and the optimal learning rate for the optimizer
is {best_hps.get('learning_rate')}.
""")

Trial 30 Complete [00h 00m 16s]
val_accuracy: 0.8539166450500488

Best val_accuracy So Far: 0.89041668176651
Total elapsed time: 00h 05m 06s

The hyperparameter search is complete. The optimal number of units in the first densely-connected
layer is 448 and the optimal learning rate for the optimizer
is 0.001.



In [17]:
#build the model with the optimal hyperparameters and train it on the data for 50 epochs 
model  = tuner.hypermodel.build(best_hps) 
history = model.fit(img_train, label_train, epochs =50, validation_split=0.2)

val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('best epoch: %d' %(best_epoch,))

Epoch 1/50


e:\programs\py\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8237 - loss: 0.4973 - val_accuracy: 0.8518 - val_loss: 0.4064
Epoch 2/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8649 - loss: 0.3708 - val_accuracy: 0.8660 - val_loss: 0.3676
Epoch 3/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8778 - loss: 0.3328 - val_accuracy: 0.8752 - val_loss: 0.3460
Epoch 4/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8861 - loss: 0.3061 - val_accuracy: 0.8798 - val_loss: 0.3354
Epoch 5/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8939 - loss: 0.2868 - val_accuracy: 0.8735 - val_loss: 0.3444
Epoch 6/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8999 - loss: 0.2708 - val_accuracy: 0.8883 - val_loss: 0.3136
Epoch 7/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9049 - loss: 0.2555 - val_accuracy: 0.8786 - val_loss: 0.3430
Epoch 8/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9076 - loss: 0.2446 - val_accurac

In [18]:
hypermodel = tuner.hypermodel.build(best_hps)

#retrain the model 
hypermodel.fit(img_train, label_train, epochs = best_epoch, validation_split = 0.2)

Epoch 1/36


e:\programs\py\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8251 - loss: 0.4932 - val_accuracy: 0.8568 - val_loss: 0.3938
Epoch 2/36
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8641 - loss: 0.3726 - val_accuracy: 0.8702 - val_loss: 0.3560
Epoch 3/36
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8770 - loss: 0.3333 - val_accuracy: 0.8732 - val_loss: 0.3473
Epoch 4/36
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8863 - loss: 0.3083 - val_accuracy: 0.8848 - val_loss: 0.3248
Epoch 5/36
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8949 - loss: 0.2862 - val_accuracy: 0.8731 - val_loss: 0.3452
Epoch 6/36
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8981 - loss: 0.2727 - val_accuracy: 0.8805 - val_loss: 0.3330
Epoch 7/36
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9031 - loss: 0.2582 - val_accuracy: 0.8821 - val_loss: 0.3314
Epoch 8/36
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9085 - loss: 0.2428 - val_accurac

In [19]:
eval_result = hypermodel.evaluate(img_test, label_test)
print("[test loss, test accuracy]:", eval_result)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 837us/step - accuracy: 0.8897 - loss: 0.4893
[test loss, test accuracy]: [0.48934951424598694, 0.8896999955177307]
